# 06 — Auswertung der Labormessungen und Abgleich der Results-Kennzahlen

Dieses Notebook deckt **alle statistischen Kennzahlen ab, die in Abschnitt 6 (Results)
der Thesis stehen**. Es hat zwei Aufgaben:

1. **Neu berechnen**, was nur in den MAFC-Arbeitsmappen steht und von keiner
   Pipeline-Stage erzeugt wird — die Aktivitäten beider Platten, die
   Nachweisgrenzen, die daraus abgeleiteten Abzählungen, die Reproduzierbarkeit
   zwischen den Platten und die prospektive Modellleistung.
2. **Gegenprüfen**, was die Pipeline schon erzeugt. Dieser Teil rechnet nichts
   neu, sondern liest die Stage-Artefakte und vergleicht sie mit den Zahlen im
   Thesis-Text. Abweichungen werden als `FAIL` ausgegeben.

Die Berechnung geht bewusst **von den Roh-ODs** aus und nicht von den
Auswertungsspalten der Arbeitsmappen. Damit ist sie unabhängig von den
Formelfehlern und den von Hand überschriebenen Zellen, die in den Blättern
vorhanden waren.

## Ausschlussregel

Ein Well geht in den Mittelwert einer Variante ein, wenn

* sein Rohwert numerisch ist (Textzellen wie `overflow` fallen heraus),
* seine blank-korrigierte OD **größer als Null** ist — Wells *auf oder unter*
  dem Reagenzienleerwert werden ausgeschlossen, und
* es im Pipettierschema einer Variante zugeordnet ist.

Diese Regel entspricht der Formulierung im Methodenteil (`subsec:methods_mafc`).

## Reihenfolge

Zellen von oben nach unten ausführen. Teil G schreibt alle Zahlen in
`ML_NylC/experiments/derived/`, damit der Thesis-Text eine Herkunftsdatei hat.

In [7]:
from pathlib import Path
import json, re, os
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import column_index_from_string as col_idx
from scipy.stats import spearmanr

# Pfade -- das Notebook liegt in Bachelorarbeit/nylon_md_features/Notebooks/
ROOT = Path(os.environ.get("BA_ROOT", Path.cwd().parents[1])).resolve()
ML   = ROOT / "ML_NylC"
RES  = ML / "results" / os.environ.get("BA_EXPERIMENT", "reference_cpu")
MAFC = ML / "experiments" / "MAFC"
OUT  = ML / "experiments" / "derived"
OUT.mkdir(parents=True, exist_ok=True)

for p in (ML, RES, MAFC):
    assert p.is_dir(), f"Pfad fehlt: {p}"
print("ROOT:", ROOT)
print("RES :", RES)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

ROOT: C:\Users\arthu\OneDrive\Dokumente\Bachelorarbeit
RES : C:\Users\arthu\OneDrive\Dokumente\Bachelorarbeit\ML_NylC\results\reference_cpu


---
## Teil A — MAFC-Arbeitsmappen einlesen

Layout der Mappen (CLARIOstar-Export, Blatt `Microplate End point`):

| Inhalt | Ort |
|---|---|
| Roh-ODs | Zeilen 15–22, Spalten B–M (Plattenreihen A–H, Wells 1–12) |
| Reagenzienleerwerte | `C21`, `D21`, `E21` (= G02–G04) |
| Steigung der Kalibriergeraden | `K24` |
| Reaktionszeit (h) | `F24` |
| Enzymmassenkonzentration (mg/L) | `H24` |
| Auswertungsblock | Zeilen 28–35, Spalten B–M |
| Variantenzeilen | Zeilen 36–63, Spalte B = Name, Spalte C = `AVERAGE(...)` |

Die Zuordnung Well → Variante steht nur implizit in den `AVERAGE`-Formeln. Sie
wird hier daraus abgeleitet und in Teil G als Layout-Tabelle exportiert, damit
künftige Läufe nicht mehr von der Formelstruktur abhängen. Zellen mit
`$H$24*2` markieren Wells, die bei doppelter Enzymmenge (50 nM) gemessen wurden.

In [8]:
PLATES = {"2h": ("051026_MAFC.xlsx", 2.0), "20h": ("061026_MAFC.xlsx", 20.0)}
SHEET = "Microplate End point"
RAW_ROW0, PLATE_ROWS, N_WELLS = 15, "ABCDEFGH", 12
BLANK_CELLS = ["C21", "D21", "E21"]
SLOPE_CELL, TIME_CELL, MASS_CELL = "K24", "F24", "H24"
CALIBRATED_MAX_UM = 2000.0          # Linearitaetsgrenze der 6-AHA-Standardkurve
NOMINAL_CONC_NM = 25.0              # Enzymkonzentration, auf die MASS_CELL sich bezieht


def _avg_cells(formula):
    # Zellbezuege einer AVERAGE-Formel als (Spaltenindex, Zeile) aufloesen
    inner = formula[formula.index("(") + 1: formula.rindex(")")]
    out = []
    for part in inner.split(","):
        part = part.replace("$", "").strip()
        if ":" in part:
            a, b = part.split(":")
            ma, mb = re.match(r"([A-Z]+)(\d+)", a), re.match(r"([A-Z]+)(\d+)", b)
            for c in range(col_idx(ma.group(1)), col_idx(mb.group(1)) + 1):
                out.append((c, int(ma.group(2))))
        else:
            m = re.match(r"([A-Z]+)(\d+)", part)
            if m:
                out.append((col_idx(m.group(1)), int(m.group(2))))
    return out


def read_plate(path, hours, label):
    wf = openpyxl.load_workbook(path, data_only=False)[SHEET]   # Formeln
    wv = openpyxl.load_workbook(path, data_only=True)[SHEET]    # Werte

    slope = wv[SLOPE_CELL].value
    mass = wv[MASS_CELL].value
    assert abs(wv[TIME_CELL].value - hours) < 1e-9, "Reaktionszeit im Blatt weicht ab"

    blank_ods = [wv[c].value for c in BLANK_CELLS]
    blank = float(np.mean(blank_ods))
    blank_sem = float(np.std(blank_ods, ddof=1) / np.sqrt(len(blank_ods)))

    layout, doubled = {}, set()
    for r in range(36, 64):
        name = wf.cell(row=r, column=2).value
        fml = wf.cell(row=r, column=3).value
        if not (isinstance(name, str) and isinstance(fml, str) and fml.startswith("=AVERAGE")):
            continue
        for c, rr in _avg_cells(fml):
            if not 28 <= rr <= 35:
                continue
            pos = f"{PLATE_ROWS[rr - 28]}{c - 1:02d}"
            layout[pos] = name
            cell = wf.cell(row=rr, column=c).value
            if isinstance(cell, str) and "*2" in cell.replace("$", ""):
                doubled.add(pos)

    rows = []
    for i, pr in enumerate(PLATE_ROWS):
        for w in range(1, N_WELLS + 1):
            pos = f"{pr}{w:02d}"
            raw = wv.cell(row=RAW_ROW0 + i, column=1 + w).value
            numeric = isinstance(raw, (int, float))
            corr = (raw - blank) if numeric else np.nan
            m = mass * 2 if pos in doubled else mass
            conc = (corr / slope) if numeric else np.nan
            rows.append(dict(
                plate=label, well=pos, variant=layout.get(pos),
                raw_od=raw if numeric else np.nan, corrected_od=corr,
                conc_um=conc, enzyme_conc_nM=NOMINAL_CONC_NM * (2 if pos in doubled else 1),
                enzyme_mass_mg_per_l=m,
                spec_act=(conc / hours / m) if numeric else np.nan,
                numeric=numeric,
                at_or_below_blank=bool(numeric and corr <= 0)))
    wells = pd.DataFrame(rows)
    wells["used"] = wells.numeric & ~wells.at_or_below_blank & wells.variant.notna()

    meta = dict(plate=label, hours=hours, slope=slope,
                enzyme_mass_mg_per_l=mass, blank_od=blank, blank_sem_od=blank_sem,
                lod_spec_act=(3 * blank_sem / slope) / hours / mass,
                calibrated_max_spec_act=(CALIBRATED_MAX_UM / hours) / mass)
    return wells, meta


wells, metas = [], []
for label, (fname, hours) in PLATES.items():
    w, m = read_plate(MAFC / fname, hours, label)
    wells.append(w)
    metas.append(m)
wells = pd.concat(wells, ignore_index=True)
plate_meta = pd.DataFrame(metas).set_index("plate")
plate_meta.T

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\arthu\\OneDrive\\Dokumente\\Bachelorarbeit\\ML_NylC\\experiments\\MAFC\\051026_MAFC.xlsx'

### Well-Bilanz

Zur Kontrolle: wie viele Wells sind einer Variante zugeordnet, wie viele fallen
wegen Textwert oder wegen OD auf/unter dem Blank heraus. Die ausgeschlossenen
Wells werden namentlich ausgegeben — diese Liste gehört in den Anhang, damit der
Ausschluss nachvollziehbar ist.

In [ ]:
bilanz = (wells.assign(zugeordnet=wells.variant.notna())
          .groupby("plate")
          .agg(wells_total=("well", "size"),
               zugeordnet=("zugeordnet", "sum"),
               nicht_numerisch=("numeric", lambda s: int((~s).sum())),
               auf_oder_unter_blank=("at_or_below_blank", "sum"),
               verwendet=("used", "sum")))
display(bilanz)

ausgeschlossen = wells[wells.variant.notna() & wells.numeric & wells.at_or_below_blank]
print("Wells auf/unter dem Blank, einer Variante zugeordnet:")
print(ausgeschlossen[["plate", "well", "variant", "corrected_od"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

In [ ]:
def aggregate(df):
    g = df[df.used].groupby(["plate", "variant"])
    out = g.agg(n=("spec_act", "size"),
                spec_act=("spec_act", "mean"),
                spec_act_sem=("spec_act", lambda s: s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan),
                conc_um=("conc_um", "mean")).reset_index()
    out["product_mM"] = out.conc_um / 1000.0
    out["loadings_nM"] = g.enzyme_conc_nM.apply(
        lambda s: ", ".join(str(int(v)) for v in sorted(s.unique()))).values
    return out


variants = aggregate(wells)
v2 = variants[variants.plate == "2h"].set_index("variant")
v20 = variants[variants.plate == "20h"].set_index("variant")

lod2, lod20 = plate_meta.loc["2h", "lod_spec_act"], plate_meta.loc["20h", "lod_spec_act"]
cal2, cal20 = plate_meta.loc["2h", "calibrated_max_spec_act"], plate_meta.loc["20h", "calibrated_max_spec_act"]
REF = ["TS", "HP"]

print(f"Nachweisgrenze (3 x SEM des Blanks):   2 h {lod2:7.1f}   20 h {lod20:7.1f}  [umol/h/mg]")
print(f"Kalibrierobergrenze (2000 uM):         2 h {cal2:7.1f}   20 h {cal20:7.1f}  [umol/h/mg]")
for lbl, tbl in (("2 h", v2), ("20 h", v20)):
    print(f"\n=== {lbl} ===")
    print(tbl.sort_values("spec_act", ascending=False)
            [["n", "spec_act", "spec_act_sem", "product_mM", "loadings_nM"]]
            .to_string(float_format=lambda v: f"{v:9.2f}"))

---
## Teil B — Abzählungen, die im Results-Text stehen

Jede Zahl, die in 6.3.1 und 6.3.2 als Abzählung oder Extremwert auftaucht, wird
hier aus den Tabellen von Teil A abgeleitet, damit sie nicht von Hand gepflegt
werden muss.

In [ ]:
des2 = v2.drop(index=[r for r in REF if r in v2.index])
des20 = v20.drop(index=[r for r in REF if r in v20.index])
within2 = des2[des2.spec_act <= cal2]

B = {
    "2h_n_entries":                      int(len(v2)),
    "2h_n_above_calibrated_range":        int((v2.spec_act > cal2).sum()),
    "2h_above_calibrated_range":          sorted(v2.index[v2.spec_act > cal2]),
    "2h_max_product_mM":                  float(v2.product_mM.max()),
    "2h_max_product_variant":             v2.product_mM.idxmax(),
    "2h_designs_above_TS_product":        int((des2.product_mM > v2.loc["TS", "product_mM"]).sum()),
    "2h_TS_product_mM":                   float(v2.loc["TS", "product_mM"]),
    "2h_designs_above_TS_specact":        int((des2.spec_act > v2.loc["TS", "spec_act"]).sum()),
    "2h_TS_specact":                      float(v2.loc["TS", "spec_act"]),
    "2h_designs_above_HP_specact":        int((des2.spec_act > v2.loc["HP", "spec_act"]).sum()),
    "2h_HP_specact":                      float(v2.loc["HP", "spec_act"]),
    "2h_best_within_calibrated":          within2.spec_act.idxmax(),
    "2h_best_within_calibrated_specact":  float(within2.spec_act.max()),
    "2h_best_within_calibrated_product":  float(within2.product_mM.max()),
    "2h_best_within_fold_vs_TS":          float(within2.spec_act.max() / v2.loc["TS", "spec_act"]),
    "2h_n_below_lod":                     int((v2.spec_act < lod2).sum()),
    "2h_lod_spec_act":                    float(lod2),
    "20h_lod_spec_act":                   float(lod20),
    "20h_blank_od":                       float(plate_meta.loc["20h", "blank_od"]),
    "20h_blank_sem_od":                   float(plate_meta.loc["20h", "blank_sem_od"]),
    "20h_design_specact_min":             float(des20.spec_act.min()),
    "20h_design_specact_min_variant":     des20.spec_act.idxmin(),
    "20h_design_specact_max":             float(des20.spec_act.max()),
    "20h_design_specact_max_variant":     des20.spec_act.idxmax(),
    "20h_design_max_product_mM":          float(des20.product_mM.max()),
    "20h_designs_above_lod":              sorted(des20.index[des20.spec_act > lod20]),
    "20h_ref_product_mM":                 {r: float(v20.loc[r, "product_mM"]) for r in REF},
}
for k, val in B.items():
    print(f"  {k:36} {val}")

NameError: name 'v2' is not defined

---
## Teil C — Reproduzierbarkeit zwischen den beiden Platten

Zwei Fragen, zwei Rangkorrelationen:

* **Pro Variante** — gibt es ein reproduzierbares probenabhängiges Signal?
* **Pro Wellposition** — erklärt die Position auf der Platte die Werte? Das
  Pipettierschema wurde zwischen den Platten absichtlich geändert, deshalb
  trennt dieser Vergleich Positions- von Probeneffekten.

Für den Positionsvergleich wird die **blank-korrigierte OD** verwendet, nicht die
spezifische Aktivität: so geht die Enzymmasse nicht ein, und es wird wirklich nur
die Position verglichen. Berücksichtigt werden Positionen, die auf **beiden**
Platten einer Variante zugeordnet sind und auf beiden einen numerischen Wert
haben.

In [ ]:
both = sorted(set(v2.index) & set(v20.index))
dec = [v for v in both if v20.loc[v, "product_mM"] < v2.loc[v, "product_mM"]]
inc = [v for v in both if v20.loc[v, "product_mM"] >= v2.loc[v, "product_mM"]]

rho_var, p_var = spearmanr(v2.loc[both, "spec_act"], v20.loc[both, "spec_act"])

assigned = wells[wells.variant.notna() & wells.numeric]
od2 = assigned[assigned.plate == "2h"].set_index("well").corrected_od
od20 = assigned[assigned.plate == "20h"].set_index("well").corrected_od
pos_common = sorted(set(od2.index) & set(od20.index))
rho_pos, p_pos = spearmanr(od2[pos_common], od20[pos_common])

C = {
    "n_variants_both_plates":   len(both),
    "n_product_decreasing":     len(dec),
    "n_product_increasing":     len(inc),
    "product_increasing":       sorted(inc),
    "rho_per_variant":          float(rho_var), "p_per_variant": float(p_var),
    "n_positions_both_plates":  len(pos_common),
    "rho_per_position":         float(rho_pos), "p_per_position": float(p_pos),
}
for k, val in C.items():
    print(f"  {k:26} {val}")

print("\nProduktvergleich je Variante [mM]:")
cmp = pd.DataFrame({"2h": v2.loc[both, "product_mM"], "20h": v20.loc[both, "product_mM"]})
cmp["Richtung"] = np.where(cmp["20h"] < cmp["2h"], "sinkt", "steigt")
print(cmp.sort_values("2h", ascending=False).to_string(float_format=lambda v: f"{v:7.3f}"))

---
## Teil D — Prospektive Modellleistung

Vorhergesagte gegen gemessene Aktivität. Die Rangkorrelation ist das primäre
Maß, weil sie unter jeder monotonen Transformation der Aktivitätsskala invariant
ist — Trainings- und Validierungsmessung teilen keine nachgewiesene gemeinsame
Skala.

Vier Varianten: gegen die 20-h- und gegen die 2-h-Messung, jeweils über alle
Konstrukte mit Vorhersage und nur über die vom $k$-DPP gezogenen Panelvarianten.

In [ ]:
pred = pd.read_csv(ML / "experiments" / "tested_variants_predictions.csv")
pred["key"] = (pred.variant_id.str.replace("NylC_", "", regex=False)
                              .str.replace("NylC-", "", regex=False))
pred = pred.dropna(subset=["predicted_activity_mean"])
is_panel = pred.panel_variant.astype(str).str.lower() == "true"

D = {}
rows = []
for lbl, tbl in (("20h", v20), ("2h", v2)):
    for scope, mask in (("alle", pd.Series(True, index=pred.index)), ("panel", is_panel)):
        sub = pred[mask & pred.key.isin(tbl.index)]
        rho, p = spearmanr(sub.predicted_activity_mean, tbl.loc[sub.key, "spec_act"].values)
        D[f"rho_{lbl}_{scope}"] = float(rho)
        D[f"p_{lbl}_{scope}"] = float(p)
        D[f"n_{lbl}_{scope}"] = int(len(sub))
        rows.append(dict(Messung=lbl, Umfang=scope, n=len(sub), rho=rho, p=p))
display(pd.DataFrame(rows).round({"rho": 3, "p": 3}))

panel = pred[is_panel]
D.update({
    "pred_min": float(pred.predicted_activity_mean.min()),
    "pred_max": float(pred.predicted_activity_mean.max()),
    "pred_mean": float(pred.predicted_activity_mean.mean()),
    "panel_pred_span": float(panel.predicted_activity_mean.max() - panel.predicted_activity_mean.min()),
    "panel_sd_calibrated_min": float(panel.predicted_activity_sd_calibrated.min()),
    "panel_sd_calibrated_max": float(panel.predicted_activity_sd_calibrated.max()),
})
measured = v20.loc[pred[pred.key.isin(v20.index)].key, "spec_act"]
D.update({"meas20_min": float(measured.min()), "meas20_max": float(measured.max()),
          "meas20_mean": float(measured.mean())})
for k, val in D.items():
    if not k.startswith(("rho", "p_", "n_")):
        print(f"  {k:26} {val:.1f}")

NameError: name 'v20' is not defined

---
## Teil E — Abdeckung der vier Zielpositionen im Trainingsdatensatz

Welche Aminosäuren an den vier Positionen im Trainingsdatensatz beobachtet
wurden und welche im getesteten Panel vorkommen. Liefert `tab:training_coverage`.

Die Mutationsangaben sind Mehrfachsubstitutionen in einem Feld
(`D99R-F134W-D304M`), deshalb wird mit `findall` über alle Substitutionen
gelesen und nicht nur über die erste.

In [ ]:
core = pd.read_csv(RES / "prepare" / "core.csv")
tested = pd.read_csv(ML / "experiments" / "tested_variants.csv")
PARENT = {99: "D", 134: "F", 304: "D", 330: "R"}

cov_rows = []
for pos, parent in PARENT.items():
    observed = {parent}
    for ms in core.mutations.dropna():
        for _wt, p_, mt in re.findall(r"([A-Z])(\d+)([A-Z])", str(ms)):
            if int(p_) == pos:
                observed.add(mt)
    used = sorted(set(tested[f"aa{pos}"].dropna().astype(str)) - {"nan", ""})
    cov_rows.append(dict(Position=pos, Parental=parent,
                         n_Training=len(observed), Training=", ".join(sorted(observed)),
                         n_Panel=len(used), Panel=", ".join(used)))
coverage_table = pd.DataFrame(cov_rows)
display(coverage_table)

# Wie viele Panelvarianten tragen an wie vielen Positionen einen im Training nie beobachteten Rest
train_obs = {r.Position: set(r.Training.replace(" ", "").split(",")) for r in coverage_table.itertuples()}
novel = []
for _, r in tested.iterrows():
    k = sum(1 for pos in PARENT
            if str(r.get(f"aa{pos}", "")) not in ("nan", "") and str(r[f"aa{pos}"]) not in train_obs[pos])
    novel.append(k)
tested = tested.assign(n_novel_positions=novel)
designs = tested[tested.variant_source != "reference_enzyme"]
E = {"n_designs": int(len(designs)),
     "novel_position_counts": designs.n_novel_positions.value_counts().sort_index().to_dict(),
     "n_with_P134": int((designs.aa134.astype(str) == "P").sum())}
print(E)

,Position,Parental,n_Training,Training,n_Panel,Panel
0,99,D,4,"D, G, R, V",6,"D, E, G, N, S, T"
1,134,F,2,"F, W",7,"A, F, I, L, P, V, Y"
2,304,D,10,"D, E, I, L, M, Q, R, S, V, W",1,L
3,330,R,3,"A, Q, R",3,"A, G, S"


{'n_designs': 22, 'novel_position_counts': {1: 1, 2: 8, 3: 13}, 'n_with_P134': 9}


---
## Teil F — Abgleich gegen die Pipeline-Artefakte

Dieser Teil rechnet **nichts** neu. Er liest die Stage-Artefakte und vergleicht
sie mit den Zahlen, die im Thesis-Text stehen. Jede Zeile ist `PASS` oder `FAIL`.

Damit ist dokumentiert, welche Kennzahlen in Results reproduzierbar sind und
welche nicht — das ist der Punkt, den der Reproduzierbarkeitsabschnitt des
Methodenteils behauptet.

In [ ]:
CHECKS = []

def check(label, thesis, computed, tol=0.05, source=""):
    if isinstance(thesis, (int, float)) and isinstance(computed, (int, float)):
        ok = abs(float(thesis) - float(computed)) <= tol
    else:
        ok = str(thesis) == str(computed)
    CHECKS.append(dict(Kennzahl=label, Thesis=thesis, Berechnet=computed,
                       Status="PASS" if ok else "FAIL", Quelle=source))

# --- 6.1.1 Nested cross-validation ---
nm = pd.read_csv(RES / "evaluate" / "nested_metrics.csv").set_index("uncertainty")
src = "evaluate/nested_metrics.csv"
check("MAE nested", 45.72, nm.loc["uncalibrated", "mae"], 0.01, src)
check("RMSE nested", 60.30, nm.loc["uncalibrated", "rmse"], 0.01, src)
check("R2 nested", 0.661, nm.loc["uncalibrated", "r2"], 0.001, src)
check("Pearson nested", 0.813, nm.loc["uncalibrated", "pearson"], 0.001, src)
check("Spearman nested", 0.805, nm.loc["uncalibrated", "spearman"], 0.001, src)
check("NLPD unkalibriert", 5.591, nm.loc["uncalibrated", "nlpd_observed"], 0.001, src)
check("NLPD sigma-skaliert", 5.581, nm.loc["sigma_scaled", "nlpd_observed"], 0.001, src)

npred = pd.read_csv(RES / "evaluate" / "nested_predictions.csv")
src = "evaluate/nested_predictions.csv"
check("Beobachtet min", 77.7, npred.observed.min(), 0.1, src)
check("Beobachtet max", 520.0, npred.observed.max(), 0.1, src)
check("Beobachtet Median", 222.5, npred.observed.median(), 0.1, src)
check("Beobachtet SD", 105.1, npred.observed.std(ddof=1), 0.1, src)
check("Medianer Absolutfehler", 33.1, npred.abs_error.median(), 0.1, src)
check("Kleinster Fehler", 2.1, npred.abs_error.min(), 0.1, src)

# --- 6.1.2 Robustheit und gewaehlte Konfiguration ---
sel = pd.read_csv(RES / "evaluate" / "selected_settings.csv")
src = "evaluate/selected_settings.csv"
check("Folds mit physical", 34, int(sel.loc[sel.selected_descriptor_set == "physical", "count"].sum()), 0, src)
check("Folds mit epistatic", 35, int(sel.loc[sel.selected_model == "epistatic", "count"].sum()), 0, src)

model = json.loads((RES / "fit" / "model.json").read_text())
hp = model["hyperparameters"]
src = "fit/model.json"
check("Laengenskalen-Multiplikator", 0.402, hp["lengthscale_multiplier"], 0.001, src)
check("sigma_main", 0.528, hp["sigma_main"], 0.001, src)
check("sigma_epi", 0.781, hp["sigma_epi"], 0.001, src)
check("sigma_noise", 0.295, hp["sigma_noise"], 0.001, src)
check("Varianzkorrektur", 1.120, model["sigma_scale"], 0.001, src)
check("Restarts konvergiert", 8, hp["successful_restarts"], 0, src)
check("Hyperparameter am Rand", False, hp["any_boundary_hit"], source=src)

eoc = pd.read_csv(RES / "evaluate" / "error_by_order_correlations.csv")
eoc = eoc[eoc.scope == "all_variants"].set_index("against")
src = "evaluate/error_by_order_correlations.csv"
check("rho Ordnung~|Fehler|", 0.13, eoc.loc["absolute_error", "spearman_rho"], 0.005, src)
check("rho Ordnung~Aktivitaet", 0.76, eoc.loc["observed_activity", "spearman_rho"], 0.005, src)
check("rho Ordnung~rel. Fehler", -0.19, eoc.loc["relative_error", "spearman_rho"], 0.005, src)

ph = pd.read_csv(RES / "evaluate" / "position_holdout_metrics.csv").set_index("heldout_position")
src = "evaluate/position_holdout_metrics.csv"
for pos, mae in ((99, 99.1), (134, 152.1), (304, 77.6), (330, 76.3)):
    check(f"Position-Holdout MAE {pos}", mae, ph.loc[pos, "mae"], 0.1, src)

# Train-Mean-Referenz ist in keinem Artefakt -- hier aus den Holdout-Vorhersagen abgeleitet
phpred = pd.read_csv(RES / "evaluate" / "position_holdout_predictions.csv")
src = "abgeleitet aus position_holdout_predictions.csv + prepare/core.csv"
for pos, tm_thesis in ((99, 70.2), (134, 152.9), (304, 92.7), (330, 125.0)):
    g = phpred[phpred.heldout_position == pos]
    train_mean = core[~core.variant_id.isin(g.variant_id)].activity_pa6.mean()
    check(f"Train-Mean-MAE {pos}", tm_thesis, np.abs(g.observed - train_mean).mean(), 0.1, src)

# --- 6.1.5 Unsicherheit ---
cov = pd.read_csv(RES / "evaluate" / "coverage.csv").set_index("interval")
src = "evaluate/coverage.csv"
for iv, unc, cal in (("50%", 0.571, 0.600), ("68%", 0.657, 0.714),
                     ("90%", 0.829, 0.829), ("95%", 0.886, 0.914)):
    check(f"Coverage {iv} unkalibriert", unc, cov.loc[iv, "uncalibrated_coverage"], 0.001, src)
    check(f"Coverage {iv} kalibriert", cal, cov.loc[iv, "calibrated_coverage"], 0.001, src)

src = "evaluate/nested_predictions.csv"
z = (npred.observed - npred.predicted) / npred.predicted_std_observed
zc = (npred.observed - npred.predicted) / npred.predicted_std_observed_calibrated
check("Mittleres sd unkalibriert", 55.4, npred.predicted_std_observed.mean(), 0.1, src)
check("Mittleres sd kalibriert", 62.1, npred.predicted_std_observed_calibrated.mean(), 0.1, src)
check("RMS stand. Residuen unkal", 1.166, np.sqrt((z ** 2).mean()), 0.001, src)
check("RMS stand. Residuen kal", 1.053, np.sqrt((zc ** 2).mean()), 0.001, src)
Z95 = 1.95996398
check("Mittlere 95%-Breite unkal", 217.2, (2 * Z95 * npred.predicted_std_observed).mean(), 0.1, src)
check("Mittlere 95%-Breite kal", 243.5, (2 * Z95 * npred.predicted_std_observed_calibrated).mean(), 0.1, src)
check("Ausserhalb 95% unkal", 4, int((z.abs() > Z95).sum()), 0, src)
check("Ausserhalb 95% kal", 3, int((zc.abs() > Z95).sum()), 0, src)
rho_sd, _ = spearmanr(npred.predicted_std_observed, npred.abs_error)
check("rho sd~|Fehler|", -0.06, rho_sd, 0.01, src)

# --- 6.2 Kandidaten und Panel ---
ssum = json.loads((RES / "select" / "summary.json").read_text())
src = "select/summary.json"
check("Kandidaten", 160, ssum["n_candidates"], 0, src)
check("Aktivitaetsschranke", 245.9, ssum["activity_floor"], 0.1, src)
check("Zulaessige Kandidaten", 30, ssum["n_eligible"], 0, src)

ranked = pd.read_csv(RES / "select" / "all_candidates_activity_ranked.csv")
mean_col = [c for c in ranked.columns if "mean" in c and "activity" in c][0]
src = "select/all_candidates_activity_ranked.csv"
check("Posterior-Mittel min", 161.2, ranked[mean_col].min(), 0.1, src)
check("Posterior-Mittel max", 307.4, ranked[mean_col].max(), 0.1, src)
check("Posterior-Mittel Median", 211.1, ranked[mean_col].median(), 0.1, src)

sl = pd.read_csv(RES / "select" / "shortlist_sensitivity_diagnostics.csv").set_index("shortlist_size")
src = "select/shortlist_sensitivity_diagnostics.csv"
for M, act in ((20, 284.3), (25, 282.0), (30, 278.3)):
    check(f"Panel-Aktivitaet Shortlist {M}", act, sl.loc[M, "mean_predicted_activity"], 0.1, src)
for M, ld in ((20, -21.63), (25, -20.36), (30, -18.07)):
    check(f"log-Determinante Shortlist {M}", ld, sl.loc[M, "log_determinant"], 0.01, src)
for M, ks in ((20, 0.464), (25, 0.460), (30, 0.455)):
    check(f"Kernel-Aehnlichkeit Shortlist {M}", ks, sl.loc[M, "mean_kernel_similarity"], 0.001, src)

ds = pd.read_csv(RES / "select" / "dpp_seed_sensitivity.csv")
src = "select/dpp_seed_sensitivity.csv"
check("Seed-Ziehungen", 100, len(ds), 0, src)
check("Panel-Aktivitaet ueber Seeds", 280.4, ds.mean_predicted_activity.mean(), 0.1, src)
check("Mittlerer Jaccard", 0.55, ds.jaccard_to_primary_panel.mean(), 0.01, src)

# --- Teil B/C/D gegen den Thesis-Text ---
src = "dieses Notebook, Teil B/C/D"
check("2 h ueber Kalibrierbereich", 7, B["2h_n_above_calibrated_range"], 0, src)
check("2 h Maximalprodukt [mM]", 3.86, B["2h_max_product_mM"], 0.01, src)
check("2 h Designs ueber TS (Produkt)", 12, B["2h_designs_above_TS_product"], 0, src)
check("2 h Designs ueber TS (spez. Akt.)", 13, B["2h_designs_above_TS_specact"], 0, src)
check("2 h Designs ueber HP (spez. Akt.)", 16, B["2h_designs_above_HP_specact"], 0, src)
check("2 h unter Nachweisgrenze", 10, B["2h_n_below_lod"], 0, src)
check("2 h Nachweisgrenze", 84.6, B["2h_lod_spec_act"], 0.1, src)
check("20 h Nachweisgrenze", 12.0, B["20h_lod_spec_act"], 0.1, src)
check("20 h Designs ueber Nachweisgrenze", 6, len(B["20h_designs_above_lod"]), 0, src)
check("Produkt sinkt von 2 h auf 20 h", 17, C["n_product_decreasing"], 0, src)
check("rho pro Variante", 0.007, C["rho_per_variant"], 0.01, src)
check("rho pro Position", 0.11, C["rho_per_position"], 0.01, src)
check("rho prospektiv 20 h (alle)", -0.13, D["rho_20h_alle"], 0.01, src)
check("rho prospektiv 20 h (Panel)", -0.12, D["rho_20h_panel"], 0.01, src)
check("rho prospektiv 2 h (alle)", 0.17, D["rho_2h_alle"], 0.01, src)

checks = pd.DataFrame(CHECKS)
n_fail = int((checks.Status == "FAIL").sum())
print(f"{len(checks)} Kennzahlen geprueft, {n_fail} Abweichungen\n")
display(checks[checks.Status == "FAIL"])

NameError: name 'B' is not defined

### Bekannte Abweichungen (Stand 10.10.2026)

Die `FAIL`-Zeilen sind **kein Rechenfehler des Notebooks**. Die Spalte `Thesis`
enthält die Zahl, wie sie derzeit im Text steht; `Berechnet` ist der aktuelle
Wert nach der Vereinheitlichung der Blank-Regel (DALS auf der 2-h-Platte, TPLG
auf der 20-h-Platte). Zu ändern ist also der Text, nicht die Rechnung:

| Stelle in der Thesis | steht | muss |
|---|---|---|
| 6.3.1 "exceeded the … of NylC-HP" | 16 | **17** |
| 7.3 "formed less product after 20 h" | seventeen | **sixteen** |
| 7.3 Ausnahmenliste | 7 Einträge | **8**, zusätzlich SFLS |
| 7.3 rho über die Proben | 0,007 (p = 0,97) | **−0,029 (p = 0,89)** |
| 7.3 rho über die Wells | 0,11 (p = 0,38) bei n = 68 | **0,052 (p = 0,74) bei n = 44** |
| 6.3.3 rho gegen 20 h, alle | −0,13 (p = 0,60) | **−0,162 (p = 0,51)** |
| 6.3.3 rho gegen 20 h, Panel | −0,12 (p = 0,66) | **−0,172 (p = 0,52)** |
| 6.3.3 rho gegen 2 h | 0,17 (p = 0,48) | **0,247 (p = 0,31)** |

Das `n = 68` des Positionsvergleichs ist aus den Arbeitsmappen nicht
rekonstruierbar. Dieses Notebook zählt Positionen, die auf **beiden** Platten
einer Variante zugeordnet sind und auf beiden einen numerischen Wert haben, und
kommt auf 44. Die im Text genannte Definition muss daran angeglichen werden.

Sobald der Text angepasst ist, die Erwartungswerte in der Prüfzelle oben
mitziehen — dann steht die Tabelle wieder auf `PASS` und dokumentiert den
Abgleich.

### Vollständige Prüftabelle

In [ ]:
display(checks)

---
## Teil G — Export

Alles, was in Results steht, in drei Dateien unter
`ML_NylC/experiments/derived/`:

* `lab_wells.csv` — jedes Well mit Rohwert, Korrektur, abgeleiteter Aktivität
  und Ausschlussgrund
* `lab_variants.csv` — die Variantenmittelwerte beider Platten
* `lab_well_layout.csv` — Zuordnung Well → Variante → Enzymkonzentration
* `results_numbers.json` — alle Kennzahlen aus Teil A–E
* `results_checks.csv` — die Prüftabelle aus Teil F

In [ ]:
wells.to_csv(OUT / "lab_wells.csv", index=False)
variants.to_csv(OUT / "lab_variants.csv", index=False)
(wells.loc[wells.variant.notna(), ["plate", "well", "variant", "enzyme_conc_nM"]]
      .sort_values(["plate", "well"])
      .to_csv(OUT / "lab_well_layout.csv", index=False))
coverage_table.to_csv(OUT / "training_coverage.csv", index=False)
checks.to_csv(OUT / "results_checks.csv", index=False)

payload = {
    "plate_meta": plate_meta.reset_index().to_dict(orient="records"),
    "counts_6_3": B,
    "plate_reproducibility": C,
    "prospective": D,
    "training_coverage": E,
    "exclusion_rule": "Well geht ein, wenn numerisch, korrigierte OD > 0 und einer Variante zugeordnet",
}
(OUT / "results_numbers.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False))
print("geschrieben nach", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name, f.stat().st_size, "bytes")